In [1]:
import numpy as np
import pandas as pd
import transformers
import torch

from tqdm import tqdm, trange
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/CAN_Research/attack-free-1.csv', delimiter=',')

In [4]:
df.head(5)

,timestamp,arbitration_id,data_field,attack
0,1.672531e+09,0C1,3000000430000004,0
1,1.672531e+09,0C5,3000000430000004,0
2,1.672531e+09,184,000200000000,0
3,1.672531e+09,1C7,065CB9A200003F,0
4,1.672531e+09,1CD,0000000000,0


In [5]:
# Ensure that the DataFrame is sorted by timestamp
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Create new columns for the features
df['f1'] = df['timestamp']
df['f2'] = df['timestamp'].shift(1)  # Last remote frame timestamp
df['f3'] = df['arbitration_id']
df['f4'] = df['arbitration_id'].shift(1)  # Previous frame ID
df['f5'] = df['arbitration_id'].shift(2)  # ID of previous of previous frame ID
df['f6'] = df['arbitration_id'].shift(3)  # ID of previous of previous of previous frame ID
df['f7'] = df['data_field'].apply(lambda x: len(x) // 2)  # Data size in the frame (assuming hex representation)

# Extracting DATA bytes
def extract_data_bytes(data_field, byte_index):
    """Extract a specific byte from a hexadecimal string."""
    start_index = byte_index * 2
    end_index = start_index + 2
    if len(data_field) >= end_index:
        return int(data_field[start_index:end_index], 16)
    return 0

for i in range(8):
    df[f'f{8+i}'] = df['data_field'].apply(lambda x, idx=i: extract_data_bytes(x, idx))

# Drop rows with NaN values resulting from shifting
df = df.dropna().reset_index(drop=True)

# Display the first few rows of the processed dataset
df.head()


,timestamp,arbitration_id,data_field,attack,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15
0,1.672531e+09,1C7,065CB9A200003F,0,1.672531e+09,1.672531e+09,1C7,184,0C5,0C1,7,6,92,185,162,0,0,63,0
1,1.672531e+09,1CD,0000000000,0,1.672531e+09,1.672531e+09,1CD,1C7,184,0C5,5,0,0,0,0,0,0,0,0
2,1.672531e+09,0F1,00020040,0,1.672531e+09,1.672531e+09,0F1,1CD,1C7,184,4,0,2,0,64,0,0,0,0
3,1.672531e+09,1E5,46FF28E00000D501,0,1.672531e+09,1.672531e+09,1E5,0F1,1CD,1C7,8,70,255,40,224,0,0,213,1
4,1.672531e+09,1F3,0000,0,1.672531e+09,1.672531e+09,1F3,1E5,0F1,1CD,2,0,0,0,0,0,0,0,0
